# 06 · Bayesian optimization: choosing the next settings

*Level: intermediate. Guide Section 10. Code: `optimize.expected_improvement`, `optimize.propose`, `optimize.run_loop`.*

**Idea.** Each packet costs ~16 minutes, so settings cannot be tried blindly. Bayesian optimization keeps a GP **surrogate** of the
objective $J(\theta) = \log\sigma_e + w\log\lambda$. It takes the next packet where the GP predicts a good value **or** is very unsure,
updates the GP, and repeats.

(This is level 2 of the project. Here it is only explained and simulated; nothing touches the hardware.)

In [ ]:
from tutorial_utils import *

from scipy.stats import norm
from ml import optimize

## 1. Expected improvement, by hand

Let $J_\text{best}$ be the best value measured so far. If the true value at θ is $J$, the improvement is $\max(0, J_\text{best} - J)$.
The GP says $J \sim \mathcal N(\mu, s^2)$, so the expected improvement is

$$\mathrm{EI} = (J_\text{best} - \mu)\,\Phi(z) + s\,\phi(z),\qquad z = \frac{J_\text{best}-\mu}{s},$$

where $\Phi$ is the standard Gaussian's cumulative distribution and $\phi$ its density.
* The first term is large where the mean is good (**exploitation**).
* The second is large where $s$ is large (**exploration**).

In [ ]:
def ei_by_hand(mu, s, best):
    z = (best - mu) / s
    return (best - mu) * norm.cdf(z) + s * norm.pdf(z)
mu, s = np.array([0.0, 0.5, 0.5, 1.0]), np.array([0.1, 0.1, 1.0, 1.0])
print("μ:", mu, "  s:", s, "  J_best = 0.2")
print("EI by hand:      ", ei_by_hand(mu, s, best=0.2).round(4))
print("optimize.py's EI:", optimize.expected_improvement(mu, s, 0.2).round(4))
print("→ a slightly worse mean with a large s (third) can beat a better mean that is certain (first).")

## 2. The loop on a 1D function

Start with 3 measurements. At every step: fit the GP, compute EI on a grid, measure where EI is largest.

In [ ]:
f = lambda t: 0.6 * np.sin(1.3 * t) + 0.08 * (t - 5) ** 2 - 0.2
grid = np.linspace(0, 10, 400)
X = np.array([0.8, 4.0, 8.8]); Jm = f(X)
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for step, ax in enumerate(axes.flat):
    gp = models.GP(restarts=2).fit(X[:, None], Jm[:, None])
    m, s = gp.predict(grid[:, None]); m, s = m[:, 0], s[:, 0]
    ei = ei_by_hand(m, np.maximum(s, 1e-9), Jm.min())
    ax.plot(grid, f(grid), "--", color="0.6"); ax.plot(grid, m, "C0"); ax.fill_between(grid, m - 2 * s, m + 2 * s, alpha=0.15)
    ax.scatter(X, Jm, color="k", zorder=3)
    ax2 = ax.twinx(); ax2.plot(grid, ei, "C3", lw=1); ax2.set_yticks([])
    nxt = grid[np.argmax(ei)]; ax.axvline(nxt, color="C3", ls=":"); ax.set_title(f"step {step + 1}: next θ = {nxt:.2f}")
    X, Jm = np.append(X, nxt), np.append(Jm, f(nxt))
plt.tight_layout(); plt.show()
print(f"best found {Jm.min():.3f} at θ = {X[np.argmin(Jm)]:.2f};  true minimum {f(grid).min():.3f} at θ = {grid[np.argmin(f(grid))]:.2f}")

## 3. A whole campaign on the toy CCD (10 settings)

`run_loop` starts with 10 random packets, then uses EI. `allowed` is the toy world's rule between parameters, so candidates that break it
are never proposed. The comparison is against packets taken at random.

In [ ]:
rng = np.random.default_rng(0)
measure = lambda th: synthetic.true_objective(th) + 0.01 * rng.normal()
plt.figure(figsize=(6.5, 3.5))
for strategy, color in (("bo", "C0"), ("random", "C1")):
    for seed in (0, 1):
        res = optimize.run_loop(measure, synthetic.LOW, synthetic.HIGH, n_initial=10, n_total=40, seed=seed,
                                allowed=synthetic.allowed, strategy=strategy)
        plt.plot(res["best_so_far"], color=color, label=strategy if seed == 0 else None)
plt.axvline(10, color="k", ls=":", lw=1); plt.xlabel("packets taken"); plt.ylabel("best J so far (lower is better)"); plt.legend(); plt.show()

## When to use it

* **Good:** each evaluation is expensive; up to ~20 parameters; noisy measurements.
* **Bad:** cheap evaluations (then just try many points); very many parameters at once (use a trust region, notebook 07).

**Try:** `xi=0.05` in `expected_improvement` (more exploration); a larger measurement noise.

Next: **07 · Better optimization strategies** (trust region, constraints, cheap traces, Pareto fronts).